<a href="https://colab.research.google.com/github/omsoni/llm-rag-work/blob/rag_deployment/Medical_Assistant_Deployment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Address Colab - Github Compatibility for nbformat

In [ ]:
import json

with open('Medical_Assistant_Deployment.ipynb', 'r', encoding='utf-8') as f:
    nb = json.load(f)

if 'widgets' in nb.get('metadata', {}):
    for widget_key in nb['metadata']['widgets']:
        if 'state' not in nb['metadata']['widgets'][widget_key]:
            nb['metadata']['widgets'][widget_key]['state'] = {}

with open('Medical_Assistant_Deployment.ipynb', 'w', encoding='utf-8') as f:
    json.dump(nb, f, indent=1)

# **Problem Statement**

## Business Context

A sales forecast is a prediction of future sales revenue based on historical data, industry trends, and the status of the current sales pipeline. Businesses use the sales forecast to estimate weekly, monthly, quarterly, and annual sales totals. A company needs to make an accurate sales forecast as it adds value across an organization and helps the different verticals to chalk out their future course of action.

Forecasting helps an organization plan its sales operations by region and provides valuable insights to the supply chain team regarding the procurement of goods and materials. An accurate sales forecast process has many benefits which include improved decision-making about the future and reduction of sales pipeline and forecast risks. Moreover, it helps to reduce the time spent in planning territory coverage and establish benchmarks that can be used to assess trends in the future.

## Objective

Objective is Serialize the model, and expose it as an API. Create a basic User Interface, Dockerize and Deploy the entire stack to Huggingface Space.

## Data Description

The Merck Manuals are medical references published by the American pharmaceutical company Merck & Co., that cover a wide range of medical topics, including disorders, tests, diagnoses, and drugs. The manuals have been published since 1899, when Merck & Co. was still a subsidiary of the German company Merck.

The manual is provided as a PDF with over 4,000 pages divided into 23 sections.

# **Installing and Importing the necessary libraries**

In [1]:
# Installation for GPU llama-cpp-python
# uncomment and run the following code in case GPU is being used
!CMAKE_ARGS="-DLLAMA_CUBLAS=on" FORCE_CMAKE=1 pip install llama-cpp-python==0.2.28 --no-cache-dir --no-deps -q

# Installation for CPU llama-cpp-python
# uncomment and run the following code in case GPU is not being used
# !CMAKE_ARGS="-DLLAMA_CUBLAS=off" FORCE_CMAKE=1 pip install llama-cpp-python==0.2.28 --force-reinstall --no-cache-dir -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 218.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
#Installing the libraries with the specified versions
!pip install numpy==2.0.2 pandas==2.2.2 scikit-learn==1.6.1 matplotlib==3.10.0 seaborn==0.13.2 joblib==1.4.2 xgboost==2.1.4 requests==2.32.4 huggingface_hub==0.34.0 pipreqs -q

**Note:**

- After running the above cell, kindly restart the notebook kernel (for Jupyter Notebook) or runtime (for Google Colab) and run all cells sequentially from the next cell.

- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in this notebook.

In [1]:
import warnings
warnings.filterwarnings("ignore")

# Libraries to help with reading and manipulating data
import numpy as np
import pandas as pd

# For splitting the dataset
from sklearn.model_selection import train_test_split

# Libaries to help with data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Removes the limit for the number of displayed columns
pd.set_option("display.max_columns", None)
# Sets the limit for the number of displayed rows
pd.set_option("display.max_rows", 100)

# To serialize the model
import joblib

# os related functionalities
import os

# API request
import requests

# for hugging face space authentication to upload files
from huggingface_hub import login, HfApi

# **Loading the dataset**

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
import os
os.chdir("/content/drive/My Drive/Colab Notebooks/Model Deployment/Medical Assistant")
os.makedirs("deployment_files", exist_ok=True)

# **Deployment - Backend**

## Flask API

The API has two endpoints on for **health check** and one for **posting data** from Streamlit user interface for creating predictions


In [19]:
%%writefile deployment_files/api.py
import os
import logging
import numpy as np
from flask import Flask, request, jsonify
import llama_cpp                          # ← add this
from llama_cpp import Llama
from sentence_transformers import CrossEncoder, SentenceTransformer
import chromadb
from chromadb.config import Settings
print(f"llama-cpp-python version: {llama_cpp.__version__}")
print(f"GPU offload supported: {llama_cpp.llama_supports_gpu_offload()}")

logging.basicConfig(level=logging.INFO)
log = logging.getLogger(__name__)

app = Flask(__name__)

# ---- Config ----
MODEL_PATH = "/home/user/app/models/llm/Meta-Llama-3-8B-Instruct-Q4_K_M.gguf"
CHROMA_PATH = "/home/user/app/db/chroma_db"   # verify matches Dockerfile COPY
COLLECTION_NAME = "medical_assistant-512.32"
EMBEDDER_NAME = "sentence-transformers/all-mpnet-base-v2"
RERANKER_NAME = "cross-encoder/ms-marco-MiniLM-L-12-v2"

N_GPU_LAYERS = int(os.getenv("N_GPU_LAYERS", "0"))
N_THREADS = int(os.getenv("N_THREADS", "2"))
TOP_K_RETRIEVE = 20
TOP_K_RERANK = 5

SAMPLING_PROFILES = {
    0.0: {"top_p": 0.95, "top_k": 10,  "max_tokens": 256},
    0.1: {"top_p": 0.9,  "top_k": 20,  "max_tokens": 256},
    0.3: {"top_p": 0.85, "top_k": 40,  "max_tokens": 512},
    0.7: {"top_p": 0.9,  "top_k": 50,  "max_tokens": 512},
    1.0: {"top_p": 0.95, "top_k": 100, "max_tokens": 1024},
}

LLAMA3_PROMPT = (
    "<|begin_of_text|>"
    "<|start_header_id|>system<|end_header_id|>\n\n{system}<|eot_id|>"
    "<|start_header_id|>user<|end_header_id|>\n\n{user}<|eot_id|>"
    "<|start_header_id|>assistant<|end_header_id|>\n\n"
)

SYSTEM_MESSAGE = """You are a medical diagnosis assistant that answers strictly using Merck Manual content.

Rules:
- Answer ONLY using the provided context. Do not use outside knowledge.
- If the answer cannot be derived from the Context, respond with "I don't know".
- The answer may not always be directly stated — use reasoning within the context.

Response format:
- Clinical Explanation
- Treatment Protocol
- Cite section titles if available
- Use bullet points throughout"""

# ---- Load models once at startup ----
log.info("Loading LLM...")
llm = Llama(
    model_path=MODEL_PATH,
    n_ctx=4096,
    n_gpu_layers=-1,
    n_threads=N_THREADS,
    verbose=True,
)

log.info("Loading embedder...")
embedder = SentenceTransformer(EMBEDDER_NAME)

log.info("Loading reranker...")
reranker = CrossEncoder(RERANKER_NAME)

log.info("Connecting to Chroma...")
chroma_client = chromadb.PersistentClient(
    path=CHROMA_PATH,
    settings=Settings(anonymized_telemetry=False),
)
collection = chroma_client.get_collection(COLLECTION_NAME)
log.info("Loaded collection with %d documents", collection.count())


# ---- Helpers ----
def rerank(query: str, docs: list[str], top_k: int = TOP_K_RERANK) -> list[str]:
    if not docs:
        return []
    pairs = [[query, doc] for doc in docs]
    scores = reranker.predict(pairs)
    top_indices = np.argsort(scores)[::-1][:top_k]
    return [docs[i] for i in top_indices]


def build_prompt(top_docs: list[str], query_text: str) -> str:
    context = "\n\n".join(top_docs) if top_docs else "(no context found)"
    user_text = f"<context>\n{context}\n</context>\n\n<question>\n{query_text}\n</question>"
    return LLAMA3_PROMPT.format(system=SYSTEM_MESSAGE, user=user_text)


def get_sampling_params(temp: float) -> dict:
    nearest = min(SAMPLING_PROFILES.keys(), key=lambda x: abs(x - temp))
    return SAMPLING_PROFILES[nearest]


# ---- Routes ----
@app.route("/health", methods=["GET"])
def health():
    return jsonify({"status": "ok", "doc_count": collection.count()}), 200


@app.route("/query", methods=["POST"])
def query():
    data = request.get_json(force=True, silent=True)
    if not data or "query" not in data:
        return jsonify({"error": "Request body must be JSON with a 'query' field"}), 400

    query_text = data["query"]
    temperature = float(data.get("temperature", 0.0))

    # 1. Retrieve
    query_embedding = embedder.encode(query_text, normalize_embeddings=True).tolist()
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=TOP_K_RETRIEVE,
        include=["documents"],
    )
    docs = results["documents"][0] if results["documents"] else []
    log.info("Retrieved %d docs", len(docs))

    if not docs:
        return jsonify({
            "answer": "No relevant context found in the knowledge base for this query.",
            "sources": [],
        }), 200

    # 2. Rerank
    top_docs = rerank(query_text, docs)

    # 3. Build prompt
    prompt = build_prompt(top_docs, query_text)
    log.debug("Prompt length: %d chars", len(prompt))

    # 4. Generate
    params = get_sampling_params(temperature)
    try:
        output = llm(
            prompt,
            temperature=temperature,
            top_p=params["top_p"],
            top_k=params["top_k"],
            max_tokens=params["max_tokens"],
            stop=["<|eot_id|>"],
        )
    except Exception as e:
        log.exception("LLM inference failed")
        return jsonify({"error": "Inference failed", "detail": str(e)}), 500

    return jsonify({
        "answer": output["choices"][0]["text"].strip(),
        "strategy": params,
        "sources": top_docs,
    })


if __name__ == "__main__":
    app.run(host="127.0.0.1", port=5000)

Overwriting deployment_files/api.py


### Response Function

## Setting up a Hugging Face Docker Space for the Backend

I created the following public space https://huggingface.co/spaces/omsoni/retail_chain_sales_forecast to deploy user interface and Flask API

# **Deployment - Frontend**

## Streamlit for Interactive UI

In [8]:
%%writefile deployment_files/streamlit_app.py

# streamlit_app.py
import streamlit as st
import requests

# ---- Config ----
API_URL = "http://localhost:5000/query"  # Change to your Flask API URL

STRATEGY_PROFILES = {
    0.0: {"name": "Deterministic", "top_p": 0.95, "top_k": 10,  "max_tokens": 256,  "use_case": "Factual, consistent answers"},
    0.1: {"name": "Conservative",  "top_p": 0.9,  "top_k": 20,  "max_tokens": 256,  "use_case": "Slight variation, still safe"},
    0.3: {"name": "Balanced",      "top_p": 0.85, "top_k": 40,  "max_tokens": 512,  "use_case": "General medical Q&A"},
    0.7: {"name": "Creative",      "top_p": 0.9,  "top_k": 50,  "max_tokens": 512,  "use_case": "Differential diagnosis"},
    1.0: {"name": "Exploratory",   "top_p": 0.95, "top_k": 100, "max_tokens": 1024, "use_case": "Brainstorming, research"},
}

TOP_P_OPTIONS = [0.85, 0.9, 0.95]
TOP_K_OPTIONS = [10, 20, 40, 50, 100]
MAX_TOKENS_OPTIONS = [256, 512, 1024]


def get_nearest_profile(temp: float):
    """Snap temperature to nearest strategy profile."""
    nearest = min(STRATEGY_PROFILES.keys(), key=lambda x: abs(x - temp))
    return nearest, STRATEGY_PROFILES[nearest]


# ---- Page setup ----
st.set_page_config(page_title="Medical RAG Assistant", page_icon="🩺", layout="wide")
st.title("🩺 Medical RAG Assistant")
st.caption("Ask medical questions; answers are grounded in retrieved context.")


# ---- Sidebar: Sampling controls ----
with st.sidebar:
    st.header("⚙️ Generation Settings")

    temperature = st.slider(
        "Temperature",
        min_value=0.0,
        max_value=1.0,
        value=0.0,
        step=0.05,
        help="0 = deterministic, 1 = exploratory"
    )

    # Show which strategy this temperature maps to
    nearest_temp, profile = get_nearest_profile(temperature)
    st.info(f"**Strategy: {profile['name']}**\n\n{profile['use_case']}")

    st.divider()

    # Allow user to override the auto-selected values
    st.subheader("Advanced (override defaults)")

    use_custom = st.checkbox("Customize top_p / top_k / max_tokens", value=False)

    if use_custom:
        top_p = st.selectbox(
            "top_p",
            options=TOP_P_OPTIONS,
            index=TOP_P_OPTIONS.index(profile["top_p"]),
            help="Nucleus sampling threshold"
        )
        top_k = st.selectbox(
            "top_k",
            options=TOP_K_OPTIONS,
            index=TOP_K_OPTIONS.index(profile["top_k"]),
            help="Sample from top K tokens"
        )
        max_tokens = st.selectbox(
            "max_tokens",
            options=MAX_TOKENS_OPTIONS,
            index=MAX_TOKENS_OPTIONS.index(profile["max_tokens"]),
            help="Maximum response length"
        )
    else:
        top_p = profile["top_p"]
        top_k = profile["top_k"]
        max_tokens = profile["max_tokens"]
        st.text(f"top_p:      {top_p}")
        st.text(f"top_k:      {top_k}")
        st.text(f"max_tokens: {max_tokens}")


# ---- Main: Query input ----
query = st.text_area(
    "Your question",
    placeholder="e.g., What are the early symptoms of Type 2 diabetes?",
    height=100
)

col1, col2 = st.columns([1, 5])
with col1:
    submit = st.button("Ask", type="primary", use_container_width=True)
with col2:
    if st.button("Clear", use_container_width=False):
        st.rerun()


if submit and query.strip():
    payload = {
        "query": query,
        "temperature": temperature,
        "top_p": top_p,
        "top_k": top_k,
        "max_tokens": max_tokens,
    }

    st.write("**Debug: sending payload**")
    st.json(payload)

    with st.spinner("Retrieving context and generating answer..."):
        try:
            response = requests.post(API_URL, json=payload, timeout=300)
            st.write(f"**Debug: HTTP status** = {response.status_code}")
            st.write(f"**Debug: response length** = {len(response.text)} chars")

            response.raise_for_status()
            data = response.json()

            st.write("**Debug: parsed JSON keys**")
            st.write(list(data.keys()))
            st.write("**Debug: answer field**")
            st.write(repr(data.get("answer")))   # repr shows empty strings clearly

        except requests.exceptions.RequestException as e:
            st.error(f"API request failed: {e}")
            st.stop()
        except ValueError as e:
            st.error(f"JSON parse failed: {e}")
            st.write("Raw response:", response.text[:1000])
            st.stop()

    # ---- Display answer ----
    st.subheader("Answer")
    answer = data.get("answer", "")
    if answer:
        st.write(answer)
    else:
        st.warning("Answer field is empty or missing")
        st.write("Full response:")
        st.json(data)

Overwriting deployment_files/streamlit_app.py


## Dependencies File

### Generate Requirements.txt from actual used Python packages

In [21]:
%%writefile deployment_files/requirements.txt
#Installing the libraries with the specified versions
flask==3.0.0
numpy>=1.24
sentence-transformers==3.0.1
chromadb==1.1.1
huggingface_hub==0.34.0
streamlit==1.40.2
requests==2.32.4
gunicorn>=21.2

Overwriting deployment_files/requirements.txt


## DockerFile

In [17]:
%%writefile deployment_files/Dockerfile
FROM nvidia/cuda:12.1.1-runtime-ubuntu22.04

# Install Python 3.11 + minimal system deps
RUN apt-get update && apt-get install -y --no-install-recommends \
        python3.11 \
        python3.11-venv \
        python3-pip \
        libgomp1 \
        curl \
    && rm -rf /var/lib/apt/lists/* \
    && update-alternatives --install /usr/bin/python python /usr/bin/python3.11 1 \
    && update-alternatives --install /usr/bin/python3 python3 /usr/bin/python3.11 1

# Run as non-root user (HF Spaces requirement)
RUN useradd -m -u 1000 user
USER user
WORKDIR /home/user/app

# Create a virtual environment
RUN python -m venv /home/user/venv
ENV PATH="/home/user/venv/bin:$PATH"
ENV VIRTUAL_ENV=/home/user/venv

# Unified HuggingFace cache
ENV HF_HOME=/home/user/app/hf_cache

# Install prebuilt CUDA wheel for llama-cpp-python (skips compilation)
RUN pip install --no-cache-dir --upgrade pip && \
    pip install --no-cache-dir \
        --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu121 \
        llama-cpp-python==0.2.90

# Install remaining Python dependencies
# Make sure llama-cpp-python is NOT in requirements.txt
COPY --chown=user requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Pre-download models at build time
RUN python -c "from huggingface_hub import hf_hub_download; \
    hf_hub_download(repo_id='bartowski/Meta-Llama-3-8B-Instruct-GGUF', \
    filename='Meta-Llama-3-8B-Instruct-Q4_K_M.gguf', \
    local_dir='/home/user/app/models/llm')"

RUN python -c "from sentence_transformers import SentenceTransformer; \
    SentenceTransformer('sentence-transformers/all-mpnet-base-v2')"

RUN python -c "from sentence_transformers import CrossEncoder; \
    CrossEncoder('cross-encoder/ms-marco-MiniLM-L-12-v2')"

# Pre-built Chroma DB
COPY --chown=user ./chroma_db /home/user/app/db/chroma_db

# App code last (preserves expensive model layers in cache on code changes)
COPY --chown=user api.py streamlit_app.py start.sh ./
RUN chmod +x start.sh

EXPOSE 7860
CMD ["./start.sh"]

Overwriting deployment_files/Dockerfile


In [24]:
%%writefile deployment_files/start.sh
#!/bin/bash
set -e

# Start Flask API in background on port 5000
echo "Starting Flask API..."
python api.py &
FLASK_PID=$!

# Wait for Flask to be ready (max 60s)
echo "Waiting for Flask to be ready..."
for i in {1..30}; do
    if curl -s http://localhost:5000/health > /dev/null 2>&1; then
        echo "Flask is ready!"
        break
    fi
    sleep 1
done

# Trap SIGTERM so both processes shut down cleanly
trap "kill $FLASK_PID; exit" SIGTERM SIGINT

# Start Streamlit in foreground on port 7860 (HF Spaces default)
echo "Starting Streamlit..."
streamlit run streamlit_app.py \
    --server.address=0.0.0.0 \
    --server.port=7860 \
    --server.headless=true \
    --browser.gatherUsageStats=false
    --server.enableXsrfProtection=false \
    --server.enableWebsocketCompression=false \
    --browser.gatherUsageStats=false
    --server.enableCORS=false \
    --server.fileWatcherType=none

Overwriting deployment_files/start.sh


In [12]:
!ls "./deployment_files"

api.py	Dockerfile  requirements.txt  start.sh	streamlit_app.py


## Uploading Files to Hugging Face Space (Streamlit Space)

In [22]:
from huggingface_hub import login, HfApi
from google.colab import userdata
access_key = userdata.get("HF_TOKEN") ## Hugging Face token created from access keys in write mode
repo_id = "omsoni/Medical_Assistant"  # Your Hugging Face space id

# Login to Hugging Face platform with the access token
login(token=access_key)

# Initialize the API
api = HfApi()

# Upload Streamlit app files stored in the folder called deployment_files
api.upload_folder(
    folder_path="/content/drive/My Drive/Colab Notebooks/Model Deployment/Medical Assistant/deployment_files",  # Local folder path in azureml
    repo_id=repo_id,  # Hugging face space id
    repo_type="space",  # Hugging face repo type "space"
)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...5f1/index_metadata.pickle: 100%|##########|  225kB /  225kB            

  ...e-4136c94245f1/header.bin: 100%|##########|   100B /   100B            

  ...e-4136c94245f1/length.bin: 100%|##########| 32.5kB / 32.5kB            

  ...36c94245f1/link_lists.bin: 100%|##########| 70.6kB / 70.6kB            

  ...6c94245f1/data_level0.bin: 100%|##########| 26.1MB / 26.1MB            

  .../chroma_db/chroma.sqlite3:  35%|###5      | 47.9MB /  136MB            

CommitInfo(commit_url='https://huggingface.co/spaces/omsoni/Medical_Assistant/commit/cedbfcee34a315204711ca4db0d992f37fce2782', commit_message='Upload folder using huggingface_hub', commit_description='', oid='cedbfcee34a315204711ca4db0d992f37fce2782', pr_url=None, repo_url=RepoUrl('https://huggingface.co/spaces/omsoni/Medical_Assistant', endpoint='https://huggingface.co', repo_type='space', repo_id='omsoni/Medical_Assistant'), pr_revision=None, pr_num=None)

### UI Integrated with Deployed model can be access at space endpoint below:

[Huggingface Space endpoint for Dockerized Forecasting Streamlit App and Final Model](https://huggingface.co/spaces/omsoni/retail_chain_sales_forecast)